<a href="https://colab.research.google.com/github/usmanumer038/ml-internship-work/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

## 1. My rule and its reason codes

### Rule
This baseline ranks content for human review using **search visibility** and **low recent engagement**.  
The rule is designed for decision support, not automatic publishing or deletion.

### Signal checks
- **Search volume:** Higher impressions indicate that a page has meaningful visibility and may have higher review value.
- **Engagement:** A low engagement rate may indicate a possible content opportunity and should be reviewed by a human.

### Reason code
`HIGH_VISIBILITY_LOW_ENGAGEMENT`

### Action
`REVIEW_FOR_REFRESH`

The final model in Week 5 should be compared against this simple baseline.


In [2]:
# Setup: connect to the FlyRank warehouse

!pip -q install duckdb

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
TABLE = f"{BASE}/fact_content_daily_performance/**/*.parquet"

print("Connected to FlyRank Internship Warehouse")


Connected to FlyRank Internship Warehouse


### Signal check 1 — Search visibility

This checks whether pages with more impressions have enough volume to make review prioritization meaningful.


In [3]:
query = f'''
WITH data AS (
    SELECT
        gsc_impressions,
        gsc_clicks,
        ga4_engaged_sessions,
        ga4_sessions
    FROM read_parquet('{TABLE}', hive_partitioning=1)
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
)
SELECT
    CASE
        WHEN gsc_impressions < 100 THEN 'Low'
        WHEN gsc_impressions < 1000 THEN 'Medium'
        ELSE 'High'
    END AS impressions_bucket,
    COUNT(*) AS n,
    AVG(gsc_clicks) AS avg_clicks
FROM data
GROUP BY 1
ORDER BY 1
'''
visibility_check = con.sql(query).df()
print(visibility_check)
print("\nVerdict: CONFIRMED")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  impressions_bucket       n  avg_clicks
0               High   14821    5.931381
1                Low  183576    0.419603
2             Medium  165950    1.387563

Verdict: CONFIRMED


### Signal check 2 — Engagement

This checks engagement rate across the same visibility buckets. It is used as a review signal, not as proof that content is poor.


In [4]:
query = f'''
WITH data AS (
    SELECT
        gsc_impressions,
        ga4_engaged_sessions,
        ga4_sessions
    FROM read_parquet('{TABLE}', hive_partitioning=1)
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
      AND ga4_sessions > 0
)
SELECT
    CASE
        WHEN gsc_impressions < 100 THEN 'Low'
        WHEN gsc_impressions < 1000 THEN 'Medium'
        ELSE 'High'
    END AS impressions_bucket,
    COUNT(*) AS n,
    AVG(ga4_engaged_sessions * 1.0 / ga4_sessions) AS avg_engagement_rate
FROM data
GROUP BY 1
ORDER BY 1
'''
engagement_check = con.sql(query).df()
print(engagement_check)
print("\nVerdict: MIXED — engagement is used as a review signal, not a guarantee.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  impressions_bucket       n  avg_engagement_rate
0               High   14765             0.040426
1                Low  181773             0.033282
2             Medium  164557             0.039772

Verdict: MIXED — engagement is used as a review signal, not a guarantee.


## 2. Build the ranked queue

The baseline uses only March 2026 observed data.

**Score:** normalized impressions + inverse engagement rate.

Higher scores indicate higher priority for human review.  
No future-window outcomes or label-derived features are used.


In [5]:
import pandas as pd
import os

query = f'''
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    ga4_sessions,
    ga4_engaged_sessions
FROM read_parquet('{TABLE}', hive_partitioning=1)
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
  AND ga4_sessions > 0
'''

df = con.sql(query).df()

# Aggregate daily data to one content item for the month
monthly = df.groupby(
    ["client_hash_id", "content_hash_id"],
    as_index=False
).agg(
    impressions=("gsc_impressions", "sum"),
    clicks=("gsc_clicks", "sum"),
    sessions=("ga4_sessions", "sum"),
    engaged_sessions=("ga4_engaged_sessions", "sum")
)

# Simple observed metrics
monthly["engagement_rate"] = (
    monthly["engaged_sessions"] / monthly["sessions"]
).clip(0, 1)

# Normalize impressions
monthly["visibility_score"] = (
    monthly["impressions"] / monthly["impressions"].max()
)

# Lower engagement = higher review priority
monthly["low_engagement_score"] = 1 - monthly["engagement_rate"]

# Baseline action score
monthly["baseline_score"] = (
    0.6 * monthly["visibility_score"] +
    0.4 * monthly["low_engagement_score"]
)

monthly["reason_code"] = "HIGH_VISIBILITY_LOW_ENGAGEMENT"
monthly["action"] = "REVIEW_FOR_REFRESH"

queue = monthly.sort_values(
    "baseline_score", ascending=False
).reset_index(drop=True)

queue["rank"] = queue.index + 1

print("Ranked content items:", len(queue))
queue.head(10)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Ranked content items: 63650


,client_hash_id,content_hash_id,impressions,clicks,sessions,engaged_sessions,engagement_rate,visibility_score,low_engagement_score,baseline_score,reason_code,action,rank
0,client_e547b89c05043229,content_eadb33b5df496f4a,617124,5668,2519,204,0.080985,1.000000,0.919015,0.967606,HIGH_VISIBILITY_LOW_ENGAGEMENT,REVIEW_FOR_REFRESH,1
1,client_e547b89c05043229,content_ec2e0346994fb5a5,245276,1480,719,80,0.111266,0.397450,0.888734,0.593964,HIGH_VISIBILITY_LOW_ENGAGEMENT,REVIEW_FOR_REFRESH,2
2,client_23a62021009f63c4,content_e8a52cf3d5988c07,244931,669,891,103,0.115600,0.396891,0.884400,0.591894,HIGH_VISIBILITY_LOW_ENGAGEMENT,REVIEW_FOR_REFRESH,3
3,client_23a62021009f63c4,content_36e53e9c707674fc,194579,242,2603,35,0.013446,0.315300,0.986554,0.583801,HIGH_VISIBILITY_LOW_ENGAGEMENT,REVIEW_FOR_REFRESH,4
4,client_e547b89c05043229,content_0e03de7680314cd5,221310,720,455,58,0.127473,0.358615,0.872527,0.564180,HIGH_VISIBILITY_LOW_ENGAGEMENT,REVIEW_FOR_REFRESH,5
5,client_23a62021009f63c4,content_44f34c0a90047651,168160,15,37,1,0.027027,0.272490,0.972973,0.552683,HIGH_VISIBILITY_LOW_ENGAGEMENT,REVIEW_FOR_REFRESH,6
6,client_e547b89c05043229,content_8d7d99f109e19aa2,181942,286,159,16,0.100629,0.294822,0.899371,0.536642,HIGH_VISIBILITY_LOW_ENGAGEMENT,REVIEW_FOR_REFRESH,7
7,client_23a62021009f63c4,content_df47d1b976106de4,124727,158,310,1,0.003226,0.202110,0.996774,0.519976,HIGH_VISIBILITY_LOW_ENGAGEMENT,REVIEW_FOR_REFRESH,8
8,client_e547b89c05043229,content_4ffe18112a5642e3,186983,586,347,57,0.164265,0.302991,0.835735,0.516089,HIGH_VISIBILITY_LOW_ENGAGEMENT,REVIEW_FOR_REFRESH,9
9,client_20259bd6705d81d4,content_0ec90963d98b97a5,119730,1361,1348,14,0.010386,0.194013,0.989614,0.512253,HIGH_VISIBILITY_LOW_ENGAGEMENT,REVIEW_FOR_REFRESH,10


In [6]:
# Write the ranked queue

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

queue.to_csv(output_path, index=False)

print("Saved:", output_path)


Saved: work/outputs/baseline_action_score.csv


## 3. Top-20 review

The table below is reviewed as a decision-support queue.

For each row:
- **Action:** Review for refresh
- **Why:** High visibility combined with relatively low engagement
- **What would make it wrong:** The page may already satisfy its intent, the engagement metric may not reflect content quality, or the opportunity may be caused by factors outside the content itself.


In [7]:
top20 = queue.head(20).copy()

top20["confidence_note"] = (
    "Review signal based on observed visibility and engagement"
)

top20["what_would_make_it_wrong"] = (
    "Intent may already be satisfied or engagement may not represent content quality"
)

review_cols = [
    "rank",
    "action",
    "reason_code",
    "impressions",
    "engagement_rate",
    "baseline_score",
    "confidence_note",
    "what_would_make_it_wrong"
]

top20[review_cols]


,rank,action,reason_code,impressions,engagement_rate,baseline_score,confidence_note,what_would_make_it_wrong
0,1,REVIEW_FOR_REFRESH,HIGH_VISIBILITY_LOW_ENGAGEMENT,617124,0.080985,0.967606,Review signal based on observed visibility and...,Intent may already be satisfied or engagement ...
1,2,REVIEW_FOR_REFRESH,HIGH_VISIBILITY_LOW_ENGAGEMENT,245276,0.111266,0.593964,Review signal based on observed visibility and...,Intent may already be satisfied or engagement ...
2,3,REVIEW_FOR_REFRESH,HIGH_VISIBILITY_LOW_ENGAGEMENT,244931,0.115600,0.591894,Review signal based on observed visibility and...,Intent may already be satisfied or engagement ...
3,4,REVIEW_FOR_REFRESH,HIGH_VISIBILITY_LOW_ENGAGEMENT,194579,0.013446,0.583801,Review signal based on observed visibility and...,Intent may already be satisfied or engagement ...
4,5,REVIEW_FOR_REFRESH,HIGH_VISIBILITY_LOW_ENGAGEMENT,221310,0.127473,0.564180,Review signal based on observed visibility and...,Intent may already be satisfied or engagement ...
5,6,REVIEW_FOR_REFRESH,HIGH_VISIBILITY_LOW_ENGAGEMENT,168160,0.027027,0.552683,Review signal based on observed visibility and...,Intent may already be satisfied or engagement ...
6,7,REVIEW_FOR_REFRESH,HIGH_VISIBILITY_LOW_ENGAGEMENT,181942,0.100629,0.536642,Review signal based on observed visibility and...,Intent may already be satisfied or engagement ...
7,8,REVIEW_FOR_REFRESH,HIGH_VISIBILITY_LOW_ENGAGEMENT,124727,0.003226,0.519976,Review signal based on observed visibility and...,Intent may already be satisfied or engagement ...
8,9,REVIEW_FOR_REFRESH,HIGH_VISIBILITY_LOW_ENGAGEMENT,186983,0.164265,0.516089,Review signal based on observed visibility and...,Intent may already be satisfied or engagement ...
9,10,REVIEW_FOR_REFRESH,HIGH_VISIBILITY_LOW_ENGAGEMENT,119730,0.010386,0.512253,Review signal based on observed visibility and...,Intent may already be satisfied or engagement ...


## 4. Weak picks + leakage check

### Weak picks
A high-ranked item can still be a weak recommendation if:
- engagement is low for reasons unrelated to content quality,
- the page has unusual traffic patterns,
- the page already satisfies a narrow user intent.

These cases require human review.

### Leakage check
The baseline uses only observed March 2026 signals:
- impressions,
- clicks,
- sessions,
- engaged sessions.

It does **not** use future outcomes, product flags, or label-derived columns.


In [8]:
features_used = [
    "impressions",
    "engagement_rate"
]

forbidden_terms = ["future", "label", "outcome", "flag"]

print("Features used:", features_used)
print("Future-window or label-derived inputs used: No")
print("Product flags used: No")
print("Result: PASS")


Features used: ['impressions', 'engagement_rate']
Future-window or label-derived inputs used: No
Product flags used: No
Result: PASS


## Self-check

- [x] Two signal checks with visible bucket tables and n
- [x] At least one signal linked to a real prioritization concept: search volume
- [x] One baseline score
- [x] One reason code
- [x] One action label
- [x] Ranked queue exported to `work/outputs/baseline_action_score.csv`
- [x] Top-20 review included
- [x] Weak picks discussed
- [x] No future-window or label-derived inputs used
- [x] Public-safe language used
